In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Path resolution (identical convention to the other two notebooks) ---
_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(f"Could not find data/raw/DHS_Districts from cwd {_cwd}.")

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

tfr_path = PROCESSED_DIR / "tfr_panel_full.parquet"
intensity_path = PROCESSED_DIR / "district_intensity_variables.parquet"

for p in [tfr_path, intensity_path]:
    print(p, "->", p.exists())

tfr_panel = pd.read_parquet(tfr_path)
district_intensity = pd.read_parquet(intensity_path)

print("TFR panel shape:", tfr_panel.shape)
print("District intensity shape:", district_intensity.shape)

# --- Merge on stable_district_id ---
merged = tfr_panel.merge(district_intensity, on="stable_district_id", how="left", suffixes=("", "_intensity"))
print("Merged shape (pre-restriction):", merged.shape)

n_missing_intensity = merged["delta_mobile_ownership"].isna().sum()
print(f"Rows with no matching intensity data: {n_missing_intensity} / {len(merged)}")

# --- Approximate per-district standard error for delta_mobile_ownership ---
# SE(delta) = sqrt(Var(p4) + Var(p5)), each Var(p) = p(1-p)/n -- standard
# binomial-proportion SE, combined in quadrature since the two rounds are
# independent samples
p4, n4 = merged["s930_mean"], merged["n_resp_nfhs4"]
p5, n5 = merged["v169a_mean"], merged["n_resp_nfhs5"]
merged["delta_mobile_ownership_se"] = np.sqrt((p4 * (1 - p4) / n4) + (p5 * (1 - p5) / n5))

# --- Restrict to the pre-period, rateable windows only ---
# year_approx <= 2015 (strictly pre-Jio; 2016 is ambiguous given only-annual
# granularity and Jio's Sept 2016 launch, so excluded from the clean pre-period)
RATEABLE_WINDOWS = {"primary", "sensitivity"}
pre_period = merged[
    merged["window"].isin(RATEABLE_WINDOWS) & (merged["year_approx"] <= 2015)
].copy()

print("\nPre-period (rateable, year_approx<=2015) shape:", pre_period.shape)
print("Year range:", pre_period["year_approx"].min(), "-", pre_period["year_approx"].max())
print("\nRows by round:")
print(pre_period["round"].value_counts())
print("\nDistricts covered:", pre_period["stable_district_id"].nunique(), "/ 575")
print("\nTercile distribution (delta_mobile_ownership_tercile):")
print(pre_period["delta_mobile_ownership_tercile"].value_counts())

pre_period.sort_values(["stable_district_id", "round", "year_approx"]).head(10)

/Users/eknoorsandhu/DHS_India_Research/data/processed/tfr_panel_full.parquet -> True
/Users/eknoorsandhu/DHS_India_Research/data/processed/district_intensity_variables.parquet -> True
TFR panel shape: (38664, 13)
District intensity shape: (575, 16)
Merged shape (pre-restriction): (38664, 28)
Rows with no matching intensity data: 0 / 38664

Pre-period (rateable, year_approx<=2015) shape: (14950, 28)
Year range: 2001.0 - 2015.0

Rows by round:
round
NFHS-4    8625
NFHS-5    6325
Name: count, dtype: int64

Districts covered: 575 / 575

Tercile distribution (delta_mobile_ownership_tercile):
delta_mobile_ownership_tercile
low     4992
high    4992
mid     4966
Name: count, dtype: int64


,round,stable_district_id,years_before_survey,window,TFR,total_weighted_births,total_women_years,n_age_bands_present,year_approx,low_cell_count,...,delta_mobile_ownership,nfhs5_mobile_level,nfhs5_internet_level,low_n_either_round,delta_mobile_ownership_tercile,nfhs5_mobile_level_tercile,nfhs5_internet_level_tercile,district_name_nfhs4_intensity,district_name_nfhs5_intensity,delta_mobile_ownership_se
15,NFHS-4,1.0,15.0,sensitivity,3.355340,22.063440,144.230683,4.0,2001.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
14,NFHS-4,1.0,14.0,sensitivity,4.527983,27.160918,149.586274,5.0,2002.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
13,NFHS-4,1.0,13.0,sensitivity,5.504054,32.136069,166.840930,5.0,2003.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
12,NFHS-4,1.0,12.0,sensitivity,3.851308,27.998202,177.521003,5.0,2004.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
11,NFHS-4,1.0,11.0,sensitivity,4.072479,30.981737,191.401163,5.0,2005.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
10,NFHS-4,1.0,10.0,sensitivity,3.588075,28.515665,203.024481,5.0,2006.0,True,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
9,NFHS-4,1.0,9.0,sensitivity,4.650935,31.228200,212.746953,6.0,2007.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
8,NFHS-4,1.0,8.0,sensitivity,4.108681,31.946656,223.684832,6.0,2008.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
7,NFHS-4,1.0,7.0,primary,3.684889,31.305178,240.393251,6.0,2009.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163
6,NFHS-4,1.0,6.0,primary,3.707135,32.789613,253.139986,6.0,2010.0,False,...,0.22746,0.709814,0.454159,False,high,high,high,kupwara,kupwara,0.044163


In [4]:
from linearmodels.panel import PanelOLS
from scipy import stats

# --- Build the panel: entity = district x round (since both rounds contribute
# separate retrospective observations for overlapping years), time = year_approx ---
panel_df = pre_period.copy()
panel_df["entity"] = panel_df["stable_district_id"].astype(str) + "_" + panel_df["round"]
panel_df = panel_df.set_index(["entity", "year_approx"])

# --- Center delta for interpretable coefficients; inverse-variance weight from
# the SE we just computed in Treatment_Int_Con.ipynb ---
panel_df["delta_c"] = panel_df["delta_mobile_ownership"] - panel_df["delta_mobile_ownership"].mean()
panel_df["iv_weight"] = 1 / (panel_df["delta_mobile_ownership_se"] ** 2)

# --- Year x delta interaction terms (event-study style), one base year omitted ---
BASE_YEAR = 2015.0
years = sorted(panel_df.index.get_level_values("year_approx").unique())
year_dummy_cols = []
for y in years:
    if y == BASE_YEAR:
        continue
    col = f"year_{int(y)}_x_delta"
    panel_df[col] = (panel_df.index.get_level_values("year_approx") == y).astype(float) * panel_df["delta_c"]
    year_dummy_cols.append(col)

mod = PanelOLS(
    panel_df["TFR"],
    panel_df[year_dummy_cols],
    weights=panel_df["iv_weight"],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
# Cluster by the TRUE underlying district (not the district x round entity),
# since NFHS-4 and NFHS-5's retrospective series for the same district are
# correlated observations of the same place, not independent
res = mod.fit(cov_type="clustered", clusters=panel_df["stable_district_id"])
print(res.summary)

# --- Joint Wald test: are all pre-period year x delta interactions jointly zero? ---
# (i.e., the actual parallel-trends test, not just eyeballing coefficients)
b = res.params[year_dummy_cols].values
V = res.cov.loc[year_dummy_cols, year_dummy_cols].values
wald_stat = b @ np.linalg.inv(V) @ b
wald_df = len(year_dummy_cols)
wald_p = 1 - stats.chi2.cdf(wald_stat, wald_df)
print(f"\nJoint Wald test (H0: all pre-period year x delta interactions = 0):")
print(f"  Wald stat = {wald_stat:.3f}, df = {wald_df}, p-value = {wald_p:.4f}")

# --- Descriptive companion table for later visualization: simple tercile means ---
# (illustrative only, per the noise caveat -- NOT the actual test)
tercile_means = (
    pre_period.groupby(["delta_mobile_ownership_tercile", "year_approx"])["TFR"]
    .mean()
    .reset_index()
)
print("\nMean TFR by delta tercile and year (descriptive only):")
print(tercile_means.pivot(index="year_approx", columns="delta_mobile_ownership_tercile", values="TFR"))

                          PanelOLS Estimation Summary                           
Dep. Variable:                    TFR   R-squared:                        0.0023
Estimator:                   PanelOLS   R-squared (Between):             -0.0003
No. Observations:               14950   R-squared (Within):               0.0014
Date:                Mon, Sep 28 2026   R-squared (Overall):             -0.0003
Time:                        21:26:53   Log-likelihood                   -8139.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2.2914
Entities:                        1150   P-value                           0.0039
Avg Obs:                       13.000   Distribution:                F(14,13772)
Min Obs:                       11.000                                           
Max Obs:                       15.000   F-statistic (robust):             2.1605
                            

In [5]:
# --- Robustness: restrict to years covered by rateable, primary-adjacent
# windows only -- specifically, drop years that ONLY NFHS-4 reaches (2001-2004),
# where NFHS-5 contributes zero observations and NFHS-4's own window
# classification already flags this period as lower-reliability ('sensitivity')
BOTH_ROUNDS_MIN_YEAR = 2005.0  # NFHS-5's earliest reachable year at year_approx<=2015

panel_df_restricted = panel_df[panel_df.index.get_level_values("year_approx") >= BOTH_ROUNDS_MIN_YEAR]
year_dummy_cols_restricted = [c for c in year_dummy_cols if int(c.split("_")[1]) >= BOTH_ROUNDS_MIN_YEAR]

mod_r = PanelOLS(
    panel_df_restricted["TFR"],
    panel_df_restricted[year_dummy_cols_restricted],
    weights=panel_df_restricted["iv_weight"],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_r = mod_r.fit(cov_type="clustered", clusters=panel_df_restricted["stable_district_id"])
print(res_r.summary)

b_r = res_r.params[year_dummy_cols_restricted].values
V_r = res_r.cov.loc[year_dummy_cols_restricted, year_dummy_cols_restricted].values
wald_stat_r = b_r @ np.linalg.inv(V_r) @ b_r
wald_df_r = len(year_dummy_cols_restricted)
wald_p_r = 1 - stats.chi2.cdf(wald_stat_r, wald_df_r)
print(f"\nJoint Wald test, both-rounds-covered years only (2005-2014, base=2015):")
print(f"  Wald stat = {wald_stat_r:.3f}, df = {wald_df_r}, p-value = {wald_p_r:.4f}")

                          PanelOLS Estimation Summary                           
Dep. Variable:                    TFR   R-squared:                        0.0024
Estimator:                   PanelOLS   R-squared (Between):             -0.0003
No. Observations:               12650   R-squared (Within):               0.0018
Date:                Mon, Sep 28 2026   R-squared (Overall):             -0.0002
Time:                        21:26:53   Log-likelihood                   -6148.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2.8145
Entities:                        1150   P-value                           0.0017
Avg Obs:                       11.000   Distribution:                F(10,11480)
Min Obs:                       11.000                                           
Max Obs:                       11.000   F-statistic (robust):             2.6949
                            

In [6]:
# --- Segment 3: Same pre-trends test using nfhs5_internet_level ---
# No differencing noise (single-round estimate), no round-coverage asymmetry
# concern in the same way -- an independent read on the same question

panel_df["internet_c"] = panel_df["nfhs5_internet_level"] - panel_df["nfhs5_internet_level"].mean()

year_dummy_cols_int = []
for y in years:
    if y == BASE_YEAR:
        continue
    col = f"year_{int(y)}_x_internet"
    panel_df[col] = (panel_df.index.get_level_values("year_approx") == y).astype(float) * panel_df["internet_c"]
    year_dummy_cols_int.append(col)

mod_int = PanelOLS(
    panel_df["TFR"],
    panel_df[year_dummy_cols_int],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_int = mod_int.fit(cov_type="clustered", clusters=panel_df["stable_district_id"])
print(res_int.summary)

b_int = res_int.params[year_dummy_cols_int].values
V_int = res_int.cov.loc[year_dummy_cols_int, year_dummy_cols_int].values
wald_stat_int = b_int @ np.linalg.inv(V_int) @ b_int
wald_df_int = len(year_dummy_cols_int)
wald_p_int = 1 - stats.chi2.cdf(wald_stat_int, wald_df_int)
print(f"\nJoint Wald test (nfhs5_internet_level, all pre-period years):")
print(f"  Wald stat = {wald_stat_int:.3f}, df = {wald_df_int}, p-value = {wald_p_int:.4f}")

# --- Diagnostic: does dropping 2009 & 2011 alone clear the delta-based test? ---
# Tells us whether this is "two anomalous years" vs. a genuine broader trend
flagged_years = [2009.0, 2011.0]
year_dummy_cols_excl = [c for c in year_dummy_cols_restricted if int(c.split("_")[1]) not in [int(y) for y in flagged_years]]

mod_excl = PanelOLS(
    panel_df_restricted["TFR"],
    panel_df_restricted[year_dummy_cols_excl],
    weights=panel_df_restricted["iv_weight"],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_excl = mod_excl.fit(cov_type="clustered", clusters=panel_df_restricted["stable_district_id"])
b_excl = res_excl.params[year_dummy_cols_excl].values
V_excl = res_excl.cov.loc[year_dummy_cols_excl, year_dummy_cols_excl].values
wald_stat_excl = b_excl @ np.linalg.inv(V_excl) @ b_excl
wald_p_excl = 1 - stats.chi2.cdf(wald_stat_excl, len(year_dummy_cols_excl))
print(f"\nJoint Wald test excluding 2009 & 2011 (delta-based):")
print(f"  Wald stat = {wald_stat_excl:.3f}, df = {len(year_dummy_cols_excl)}, p-value = {wald_p_excl:.4f}")

                          PanelOLS Estimation Summary                           
Dep. Variable:                    TFR   R-squared:                        0.0187
Estimator:                   PanelOLS   R-squared (Between):              0.0077
No. Observations:               14950   R-squared (Within):               0.0123
Date:                Mon, Sep 28 2026   R-squared (Overall):              0.0079
Time:                        21:29:37   Log-likelihood                   -7985.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      18.773
Entities:                        1150   P-value                           0.0000
Avg Obs:                       13.000   Distribution:                F(14,13772)
Min Obs:                       11.000                                           
Max Obs:                       15.000   F-statistic (robust):             8.2495
                            